# Sri Lankan House Price Estimation with Gradient Boosting and Explainable ML

**Problem.** House prices in Sri Lanka are set by seller judgement and broker opinion, so buyers struggle to tell whether an asking price is fair. This project predicts the listing price of a house from its features, giving buyers and sellers a data-driven reference.

**Dataset.** Sri Lanka Property Ads Dataset (Kaggle, ikman.lk listings, Sep 2022 – Apr 2023). Only *Houses For Sale* are used.

**Task.** Regression (target: listing price in LKR).

**Notebook outline**
1. Setup
2. Load and filter
3. Feature extraction
4. Cleaning
5. Exploratory data analysis
6. Train/test split
7. Preprocessing
8. Models with default parameters
9. Hyperparameter tuning
10. Stacking ensemble
11. Evaluation and model selection
12. Explainability: SHAP
13. Explainability: LIME
14. Interpretation
15. Save the model and test inference
16. Front-end application

## 1. Setup

In [ ]:
import ast, re, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import optuna, shap
from lime.lime_tabular import LimeTabularExplainer

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.dpi"] = 110

DATA_PATH = "../data/houses_sample.csv"
RANDOM_STATE = 42
N_TRIALS = 25      # Optuna trials per model (lower it if tuning is slow)
CV_FOLDS = 3

## 2. Load and filter
The raw file mixes rentals, land and apartments. We keep only **Houses For Sale**.

In [ ]:
raw = pd.read_csv(DATA_PATH)
print("Raw shape:", raw.shape)
print(raw["category"].value_counts())

df = raw[raw["category"] == "Houses For Sale"].copy()
print("\nHouses for sale:", len(df))

## 3. Feature extraction
Key attributes are stored as text, so they need parsing:
- `price` text such as "Rs 28,500,000" → number
- `properties` dictionary → bedrooms, bathrooms, house size, land size
- `area` dictionary → district
- Title and description → storeys, road width and amenity keywords

In [ ]:
def to_number(s):
    if pd.isna(s):
        return np.nan
    digits = re.sub(r"[^0-9.]", "", str(s))
    return float(digits) if digits else np.nan

def count_value(s):
    if pd.isna(s):
        return np.nan
    s = str(s).replace("+", "")
    return float(s) if s.replace(".", "").isdigit() else np.nan

def land_to_perches(s):
    if pd.isna(s):
        return np.nan
    value = to_number(s)
    return value * 160 if "acre" in str(s).lower() else value   # 1 acre = 160 perches

props = df["properties"].apply(ast.literal_eval).apply(pd.Series)

data = pd.DataFrame({
    "price": df["price"].apply(to_number),
    "bedrooms": props["Bedrooms"].apply(count_value),
    "bathrooms": props["Bathrooms"].apply(count_value),
    "house_size_sqft": props["House size"].apply(to_number),
    "land_size_perches": props["Land size"].apply(land_to_perches),
    "district": df["area"].apply(lambda s: ast.literal_eval(s)["name"]),
    "city": df["location"],
    "user": df["user"],
    "title": df["title"],
}, index=df.index)

data.head()

Text features: listings often mention storeys, road width and amenities that affect value but are not in the structured fields.

In [ ]:
text = (df["title"].fillna("") + " " + df["ad_description"].fillna("")).str.lower()

WORD_NUM = {"one": 1, "single": 1, "two": 2, "double": 2, "three": 3, "four": 4}
def extract_storeys(t):
    m = re.search(r"\b(\d|one|single|two|double|three|four)\s*-?\s*(?:story|storey|stories|storeys|floor)", t)
    if not m:
        return np.nan
    v = m.group(1)
    v = int(v) if v.isdigit() else WORD_NUM[v]
    return v if 1 <= v <= 5 else np.nan

def extract_road_width(t):
    m = re.search(r"(\d{1,2})\s*(?:ft|feet|foot)\s*(?:wide\s*)?(?:road|rd|carpet|access)", t)
    if not m:
        return np.nan
    v = int(m.group(1))
    return v if 6 <= v <= 60 else np.nan

KEYWORDS = {
    "kw_luxury": r"luxur",
    "kw_brand_new": r"brand\s*new|newly built",
    "kw_clear_deed": r"clear\s*(?:deed|title)",
    "kw_pool": r"swimming|pool",
    "kw_ac": r"a/c|air\s*condition|\bac\b",
    "kw_solar": r"solar",
    "kw_parking": r"parking|garage|car\s*park|carport",
    "kw_main_road": r"main\s*road",
}

data["storeys"] = text.apply(extract_storeys)
data["road_width_ft"] = text.apply(extract_road_width)
for col, pattern in KEYWORDS.items():
    data[col] = text.str.contains(pattern, regex=True).astype(int)

data[["storeys", "road_width_ft"] + list(KEYWORDS)].describe().T[["count", "mean", "min", "max"]]

## 4. Cleaning
- Drop rows with no price
- Remove duplicate ads (same title, price and city)
- Remove impossible values that are typing errors (e.g. Rs 9,000 houses, 111,111 sqft homes)
- Group rare cities (fewer than 15 ads) into "Other" so the model is not trained on near-empty categories

In [ ]:
before = len(data)
data = data.dropna(subset=["price"])
data = data.drop_duplicates(subset=["title", "price", "city"])

bounds = {
    "price": (2_000_000, 1_500_000_000),
    "house_size_sqft": (300, 15_000),
    "land_size_perches": (2, 160),
    "bedrooms": (1, 10),
    "bathrooms": (1, 10),
}
for col, (lo, hi) in bounds.items():
    data = data[data[col].between(lo, hi)]

city_counts = data["city"].value_counts()
data["city"] = data["city"].where(data["city"].map(city_counts) >= 15, "Other")

print(f"Rows: {before} -> {len(data)} ({before - len(data)} removed)")
print("Cities kept:", data["city"].nunique(), "| Districts:", data["district"].nunique())

## 5. Exploratory data analysis

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data["price"] / 1e6, bins=60, ax=ax[0])
ax[0].set(title="Price is heavily right-skewed", xlabel="Price (Rs million)")
sns.histplot(np.log1p(data["price"]), bins=60, ax=ax[1])
ax[1].set(title="Log price is close to normal", xlabel="log(1 + price)")
plt.tight_layout(); plt.show()

A few luxury homes stretch the price scale. Modelling **log(price)** stops these from dominating the loss, so the target is log-transformed.

In [ ]:
num_cols = ["bedrooms", "bathrooms", "house_size_sqft", "land_size_perches", "storeys", "road_width_ft"]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flat, num_cols):
    sns.histplot(data[col].dropna(), bins=30, ax=ax)
    ax.set_title(f"{col} (missing {data[col].isna().mean():.0%})")
plt.tight_layout(); plt.show()

Structured fields are complete. Storeys and road width come from free text, so many values are missing; tree models handle missing values well, and the gaps are imputed in preprocessing.

In [ ]:
corr = data[num_cols + ["price"]].assign(log_price=np.log1p(data["price"])).drop(columns="price").corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0)
plt.title("Correlation with log price"); plt.show()

Bathrooms, house size and bedrooms correlate most with price, and with each other. Land size shows almost no linear correlation, because small urban plots are often expensive, so its effect depends on location. Tree-based models capture this kind of interaction and are not harmed by overlapping features.

In [ ]:
top = data.groupby("district")["price"].agg(["median", "count"]).query("count >= 20").sort_values("median")
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].barh(top.index, top["median"] / 1e6)
ax[0].set(title="Median price by district (≥20 ads)", xlabel="Rs million")
data["district"].value_counts().head(10).sort_values().plot.barh(ax=ax[1])
ax[1].set(title="Listings by district (top 10)", xlabel="Ads")
plt.tight_layout(); plt.show()

Location has a strong effect on price, and the data is concentrated in Colombo and Gampaha. Predictions for other districts will be less certain, which is noted as a limitation.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sample = data.sample(min(4000, len(data)), random_state=RANDOM_STATE)
ax[0].scatter(sample["house_size_sqft"], sample["price"] / 1e6, s=6, alpha=0.35)
ax[0].set(yscale="log", title="Larger houses cost more", xlabel="House size (sqft)", ylabel="Price (Rs million, log scale)")
sns.boxplot(data=data, x="bedrooms", y=data["price"] / 1e6, ax=ax[1], showfliers=False)
ax[1].set(title="Price by bedrooms", ylabel="Rs million")
plt.tight_layout(); plt.show()

In [ ]:
kw_effect = pd.DataFrame({
    kw: [data.loc[data[kw] == 1, "price"].median() / data.loc[data[kw] == 0, "price"].median(), data[kw].mean()]
    for kw in KEYWORDS
}, index=["median_price_ratio", "share_of_ads"]).T.sort_values("median_price_ratio")
kw_effect.round(2)

A ratio above 1 means ads mentioning the keyword have higher median prices. These raw ratios mix effects (luxury homes are also bigger), so SHAP is used later to separate each feature's contribution.

In [ ]:
ads_per_user = data["user"].value_counts()
print("Sellers:", ads_per_user.size, "| Top 5 sellers' ad counts:", ads_per_user.head().tolist())
print(f"Top 10 sellers post {ads_per_user.head(10).sum() / len(data):.0%} of all ads")

**Key EDA findings → decisions**
1. Price is right-skewed → model log(price).
2. Size and room counts drive price but overlap, and land size depends on location → use tree-based models, which handle correlated features and interactions.
3. Location matters strongly and city has many categories → CatBoost (native categorical handling) is a natural candidate.
4. Text-derived features are often missing → impute for RF; boosting models can use missing values directly.
5. A few agents post many near-identical ads → split train/test **by seller** so the test score is honest.

## 6. Train/test split
A group split by seller keeps all of a seller's ads on one side, preventing near-duplicate ads from leaking into the test set.

In [ ]:
FEATURES_NUM = ["bedrooms", "bathrooms", "house_size_sqft", "land_size_perches", "storeys", "road_width_ft"] + list(KEYWORDS)
FEATURES_CAT = ["district", "city"]
FEATURES = FEATURES_NUM + FEATURES_CAT

X = data[FEATURES].copy()
y = np.log1p(data["price"])
groups = data["user"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
g_train = groups.iloc[train_idx]

cv_splits = list(GroupKFold(n_splits=CV_FOLDS).split(X_train, y_train, g_train))
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 7. Preprocessing
- **Numeric:** median imputation. No scaling, because tree models split on thresholds and are unaffected by feature scale.
- **Categorical:** ordinal encoding for RF, XGBoost and LightGBM. CatBoost receives the raw text categories and encodes them internally with ordered target statistics, which avoids target leakage.

In [ ]:
preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), FEATURES_NUM),
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), FEATURES_CAT),
], verbose_feature_names_out=False).set_output(transform="pandas")

def tree_pipeline(model):
    return Pipeline([("prep", preprocess), ("model", model)])

def catboost_model(**params):
    return CatBoostRegressor(cat_features=tuple(FEATURES_CAT), random_seed=RANDOM_STATE, verbose=0, thread_count=-1, **params)

## 8. Models with default parameters
These scores are the reference point for tuning. Random Forest is the baseline.

In [ ]:
def evaluate(model, name):
    pred_log = model.predict(X_test)
    actual, pred = np.expm1(y_test), np.expm1(pred_log)
    return {
        "model": name,
        "RMSE (log)": np.sqrt(mean_squared_error(y_test, pred_log)),
        "R2 (log)": r2_score(y_test, pred_log),
        "MAE (Rs M)": mean_absolute_error(actual, pred) / 1e6,
        "MAPE (%)": mean_absolute_percentage_error(actual, pred) * 100,
    }

default_models = {
    "Random Forest": tree_pipeline(RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)),
    "XGBoost": tree_pipeline(XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
    "LightGBM": tree_pipeline(LGBMRegressor(random_state=RANDOM_STATE, verbose=-1)),
    "CatBoost": catboost_model(),
}
default_results = []
for name, model in default_models.items():
    model.fit(X_train, y_train)
    default_results.append(evaluate(model, name))
pd.DataFrame(default_results).set_index("model").round(3)

## 9. Hyperparameter tuning
Optuna searches each model's main parameters, scoring every candidate with **group-aware cross-validation** on the training set only. The test set stays untouched until the final evaluation.

In [ ]:
def cv_rmse(model):
    scores = cross_val_score(model, X_train, y_train, cv=cv_splits, scoring="neg_root_mean_squared_error")
    return -scores.mean()

def rf_space(t):
    return tree_pipeline(RandomForestRegressor(
        n_estimators=t.suggest_int("n_estimators", 200, 600, step=100),
        max_depth=t.suggest_int("max_depth", 8, 30),
        min_samples_leaf=t.suggest_int("min_samples_leaf", 1, 10),
        max_features=t.suggest_float("max_features", 0.3, 1.0),
        random_state=RANDOM_STATE, n_jobs=-1))

def xgb_space(t):
    return tree_pipeline(XGBRegressor(
        n_estimators=t.suggest_int("n_estimators", 300, 1500, step=100),
        learning_rate=t.suggest_float("learning_rate", 0.01, 0.2, log=True),
        max_depth=t.suggest_int("max_depth", 3, 10),
        min_child_weight=t.suggest_float("min_child_weight", 1, 20, log=True),
        subsample=t.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=t.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_lambda=t.suggest_float("reg_lambda", 1e-3, 10, log=True),
        random_state=RANDOM_STATE, n_jobs=-1))

def lgbm_space(t):
    return tree_pipeline(LGBMRegressor(
        n_estimators=t.suggest_int("n_estimators", 300, 1500, step=100),
        learning_rate=t.suggest_float("learning_rate", 0.01, 0.2, log=True),
        num_leaves=t.suggest_int("num_leaves", 15, 127),
        min_child_samples=t.suggest_int("min_child_samples", 5, 60),
        subsample=t.suggest_float("subsample", 0.6, 1.0), subsample_freq=1,
        colsample_bytree=t.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_lambda=t.suggest_float("reg_lambda", 1e-3, 10, log=True),
        random_state=RANDOM_STATE, verbose=-1))

def cat_space(t):
    return catboost_model(
        iterations=t.suggest_int("iterations", 300, 1500, step=100),
        learning_rate=t.suggest_float("learning_rate", 0.02, 0.2, log=True),
        depth=t.suggest_int("depth", 4, 10),
        l2_leaf_reg=t.suggest_float("l2_leaf_reg", 1, 20, log=True))

# Each search starts from a sensible default-like setting, so tuning can only improve on it
START_POINTS = {
    "Random Forest": {"n_estimators": 300, "max_depth": 30, "min_samples_leaf": 1, "max_features": 1.0},
    "XGBoost": {"n_estimators": 300, "learning_rate": 0.1, "max_depth": 6, "min_child_weight": 1, "subsample": 1.0, "colsample_bytree": 1.0, "reg_lambda": 1.0},
    "LightGBM": {"n_estimators": 300, "learning_rate": 0.1, "num_leaves": 31, "min_child_samples": 20, "subsample": 1.0, "colsample_bytree": 1.0, "reg_lambda": 0.001},
    "CatBoost": {"iterations": 1000, "learning_rate": 0.06, "depth": 6, "l2_leaf_reg": 3},
}
spaces = {"Random Forest": rf_space, "XGBoost": xgb_space, "LightGBM": lgbm_space, "CatBoost": cat_space}
tuned, cv_scores, best_params = {}, {}, {}

for name, space in spaces.items():
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.enqueue_trial(START_POINTS[name])
    study.optimize(lambda t: cv_rmse(space(t)), n_trials=N_TRIALS)
    best_params[name] = study.best_params
    cv_scores[name] = study.best_value
    tuned[name] = space(optuna.trial.FixedTrial(study.best_params)).fit(X_train, y_train)
    print(f"{name:14s} CV RMSE (log): {study.best_value:.4f}  {study.best_params}")

## 10. Stacking ensemble
The three tuned boosting models are combined. Their out-of-fold predictions (using the same seller-grouped folds) feed a simple **meta-learner** that learns how much to trust each model.

RidgeCV is used only as the combiner of three predictions, not as a candidate model. It is a standard choice because it is stable and hard to overfit.

In [ ]:
stack = StackingRegressor(
    estimators=[("xgb", tuned["XGBoost"]), ("lgbm", tuned["LightGBM"]), ("cat", tuned["CatBoost"])],
    final_estimator=RidgeCV(),
    cv=cv_splits, n_jobs=1,
)
stack.fit(X_train, y_train)
tuned["Stacking"] = stack
print("Meta-learner weights:", dict(zip(["XGBoost", "LightGBM", "CatBoost"], stack.final_estimator_.coef_.round(3))))

## 11. Evaluation and model selection
All models are scored on the same held-out test sellers. Stacking has no Optuna CV score because it is built from the already-tuned models. Errors in rupees (MAE, MAPE) are what a buyer would actually feel.

In [ ]:
results = pd.DataFrame([evaluate(m, n) for n, m in tuned.items()]).set_index("model")
results["CV RMSE (log)"] = pd.Series(cv_scores)
comparison = results.join(pd.DataFrame(default_results).set_index("model")[["RMSE (log)"]].rename(columns={"RMSE (log)": "Default RMSE (log)"}))
comparison = comparison.sort_values("RMSE (log)")
comparison.round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
comparison[["Default RMSE (log)", "RMSE (log)"]].rename(columns={"RMSE (log)": "Tuned RMSE (log)"}).plot.bar(ax=ax[0], rot=0)
ax[0].set(title="Test RMSE: default vs tuned (lower is better)", ylabel="RMSE (log price)")
comparison["MAPE (%)"].plot.bar(ax=ax[1], rot=0, color="tab:orange")
ax[1].set(title="Test MAPE (%)", ylabel="%")
plt.tight_layout(); plt.show()

In [ ]:
best_name = comparison.index[0]
final_model = tuned[best_name]
print("Selected model:", best_name)

pred = np.expm1(final_model.predict(X_test)); actual = np.expm1(y_test)
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].scatter(actual / 1e6, pred / 1e6, s=6, alpha=0.4)
lims = [actual.min() / 1e6, actual.max() / 1e6]
ax[0].plot(lims, lims, "r--"); ax[0].set(xscale="log", yscale="log", title=f"{best_name}: predicted vs actual", xlabel="Actual (Rs M)", ylabel="Predicted (Rs M)")
sns.histplot(np.log(pred / actual), bins=60, ax=ax[1]); ax[1].set(title="Log error distribution", xlabel="log(predicted / actual)")
plt.tight_layout(); plt.show()

**Why this model is selected**
- It has the lowest test error, and its cross-validation score agrees, so the result is not a lucky split.
- The gradient boosting models beat the Random Forest baseline because they build trees sequentially, each correcting the previous errors, which captures the interaction between size and location better.
- CatBoost's native handling of `city` and `district` uses location information without creating hundreds of one-hot columns.
- If Stacking wins, it combines the strengths of all three boosters. If a single booster wins by a negligible margin, prefer it for simplicity and faster inference.
- Errors are roughly centred around zero, so the model does not systematically over- or under-price houses.

## 12. Explainability: SHAP
SHAP measures how much each feature pushes a prediction above or below the average price.

If the selected model is the stack, SHAP is computed on CatBoost, its strongest component, because TreeExplainer needs a single tree model. LIME in the next section explains the selected model directly.

In [ ]:
shap_name = best_name if best_name != "Stacking" else "CatBoost"
shap_model = tuned[shap_name]
X_shap = X_test.sample(min(1000, len(X_test)), random_state=RANDOM_STATE)

if shap_name == "CatBoost":
    explainer = shap.TreeExplainer(shap_model)
    X_view = X_shap
    shap_values = explainer(X_view)
else:
    explainer = shap.TreeExplainer(shap_model.named_steps["model"])
    X_view = shap_model.named_steps["prep"].transform(X_shap)
    shap_values = explainer(X_view)
print("SHAP computed on:", shap_name)

In [ ]:
shap_importance = pd.Series(np.abs(shap_values.values).mean(axis=0), index=X_view.columns).sort_values(ascending=False)
print(shap_importance.round(3).head(10))
shap.plots.beeswarm(shap_values, max_display=15, show=False)
plt.title("SHAP summary: feature impact on log price"); plt.show()
shap.plots.bar(shap_values, max_display=15, show=False)
plt.title("Mean |SHAP| feature importance"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, feat in zip(axes, ["house_size_sqft", "land_size_perches", "bathrooms"]):
    shap.plots.scatter(shap_values[:, feat], ax=ax, show=False)
    ax.set_title(f"Effect of {feat}")
plt.tight_layout(); plt.show()

In [ ]:
i = 0
print(f"Actual price: Rs {np.expm1(y_test.loc[X_shap.index[i]]):,.0f}")
shap.plots.waterfall(shap_values[i], max_display=12, show=False)
plt.title("Why the model priced this house as it did"); plt.show()

## 13. Explainability: LIME
LIME fits a simple local model around one house to show which features drove that specific prediction. It works on any model, including the stack.

Categories are converted to numeric codes for LIME and decoded back before prediction.

In [ ]:
cat_encoder = OrdinalEncoder().fit(X_train[FEATURES_CAT])
cat_names = {FEATURES.index(c): list(cats) for c, cats in zip(FEATURES_CAT, cat_encoder.categories_)}

def encode_for_lime(frame):
    out = frame[FEATURES].copy()
    out[FEATURES_NUM] = out[FEATURES_NUM].fillna(X_train[FEATURES_NUM].median())
    out[FEATURES_CAT] = cat_encoder.transform(frame[FEATURES_CAT])
    return out.values.astype(float)

def lime_predict(arr):
    frame = pd.DataFrame(arr, columns=FEATURES)
    for c in FEATURES_CAT:
        idx = FEATURES.index(c)
        frame[c] = [cat_names[idx][int(round(v))] for v in frame[c]]
    return final_model.predict(frame[FEATURES])

lime_explainer = LimeTabularExplainer(
    encode_for_lime(X_train), feature_names=FEATURES, mode="regression",
    categorical_features=list(cat_names), categorical_names=cat_names,
    discretize_continuous=True, random_state=RANDOM_STATE)

for i in [0, 1]:
    row = X_shap.iloc[[i]]
    exp = lime_explainer.explain_instance(encode_for_lime(row)[0], lime_predict, num_features=8)
    print(f"House {i}: actual Rs {np.expm1(y_test.loc[row.index[0]]):,.0f} | predicted Rs {np.expm1(final_model.predict(row)[0]):,.0f}")
    exp.as_pyplot_figure(); plt.title(f"LIME explanation – house {i}"); plt.tight_layout(); plt.show()

**SHAP vs LIME**
- SHAP gives consistent global and local attributions, based on game theory.
- LIME gives a quick local approximation. Its explanations can vary between runs, because it relies on random sampling.
- For the same house, both methods should agree on the main drivers (size, location, bathrooms). Where they agree, the explanation is trustworthy.

## 14. Interpretation
Based on the SHAP results (check them against your own run):
- **Location is the biggest driver overall.** City and district together contribute more than any single structural feature. The same house is priced very differently in Colombo suburbs than in outer districts.
- **Bathrooms, house size and land size** are the main structural drivers. House size shows diminishing returns: each extra square foot adds less at the top end.
- **Bathrooms outweigh bedrooms.** Extra bathrooms signal a higher-spec house, while bedroom count overlaps with size.
- **"Luxury" in the ad text carries a real premium** even after controlling for size and location, so the text features add useful signal.
- **Storeys and road width matter less**, partly because they are missing for most ads.
- **Limitations:** these are asking prices, not sale prices. The data is concentrated in the Western Province, and land factors such as road access and plot shape are only partly captured. A typical error of about 25% (MAPE) means the model gives a reasonable price range, not an exact valuation.

## 15. Save the model and test inference
The whole pipeline (preprocessing + model) is saved, so the app only needs raw inputs.

In [ ]:
joblib.dump(final_model, "house_price_model.pkl")

meta = {
    "model_name": best_name,
    "features_num": FEATURES_NUM,
    "features_cat": FEATURES_CAT,
    "keywords": list(KEYWORDS),
    "districts": sorted(data["district"].unique().tolist()),
    "cities_by_district": {d: sorted(g["city"].unique().tolist()) for d, g in data.groupby("district")},
    "test_mape": round(float(comparison.loc[best_name, "MAPE (%)"]), 1),
}
with open("model_meta.json", "w") as f:
    json.dump(meta, f, indent=2)
print("Saved house_price_model.pkl and model_meta.json")

In [ ]:
loaded = joblib.load("house_price_model.pkl")

def predict_price(house: dict) -> float:
    row = pd.DataFrame([{f: house.get(f, np.nan) for f in FEATURES}])
    for kw in KEYWORDS:
        row[kw] = int(house.get(kw, 0))
    return float(np.expm1(loaded.predict(row[FEATURES])[0]))

example = {"bedrooms": 4, "bathrooms": 3, "house_size_sqft": 2500, "land_size_perches": 10,
           "district": "Colombo", "city": "Maharagama", "storeys": 2, "kw_parking": 1}
print(f"Estimated price: Rs {predict_price(example):,.0f}")

## 16. Front-end application
`app.py` is a Streamlit app that loads `house_price_model.pkl`, takes user inputs and returns a price estimate with a SHAP explanation.

Run it with:
```
pip install -r requirements.txt
streamlit run app.py
```

**Prompt used to generate the front end (ChatGPT/Gemini):**
> Build a Streamlit app that loads a scikit-learn/CatBoost pipeline saved as `house_price_model.pkl` and metadata in `model_meta.json`. Inputs: district (dropdown), city (dropdown filtered by district), bedrooms, bathrooms, house size in sqft, land size in perches, storeys, road width in feet, and checkboxes for luxury, brand new, clear deed, pool, A/C, solar, parking, main road. The model predicts log(1+price), so convert with expm1. Show the estimated price in LKR with an approximate range based on the model's test MAPE, plus a SHAP waterfall chart explaining the prediction.

*Replace this with the exact prompt(s) you used.*